# AllenSDK-Derived Parquet Demo From Google Drive

Standalone Colab notebook for loading a built Allen Visual Behavior analysis parquet from Google Drive, checking the Neuromatch-compatible schema, summarizing the dataset, and plotting example traces.

Expected default Drive path:

```text
/content/drive/MyDrive/nma-project/data/processed/allen_visp_sst_vip_slc17a7_pilot.parquet
```

Change `PARQUET_PATH` below if your file is stored somewhere else.

In [ ]:
import importlib.util
import subprocess
import sys


def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            pip_name or import_name,
        ])


ensure_package("pyarrow")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

In [ ]:
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path.cwd()

PARQUET_PATH = DRIVE_ROOT / "nma-project/data/processed/allen_visp_sst_vip_slc17a7_pilot.parquet"

fallback_paths = [
    DRIVE_ROOT / "nma-project/data/processed/allen_sst_smoke.parquet",
    Path("data/processed/allen_visp_sst_vip_slc17a7_pilot.parquet"),
    Path("data/processed/allen_sst_smoke.parquet"),
]

if not PARQUET_PATH.exists():
    for candidate in fallback_paths:
        if candidate.exists():
            PARQUET_PATH = candidate
            break

if not PARQUET_PATH.exists():
    raise FileNotFoundError(
        "Could not find the derived parquet. Set PARQUET_PATH to the file in Google Drive. "
        f"Last checked: {PARQUET_PATH}"
    )

print(f"Using parquet: {PARQUET_PATH}")

In [ ]:
parquet_file = pq.ParquetFile(PARQUET_PATH)
parquet_columns = parquet_file.schema_arrow.names

print(f"Rows: {parquet_file.metadata.num_rows:,}")
print(f"Columns: {len(parquet_columns):,}")
print(f"Row groups: {parquet_file.metadata.num_row_groups:,}")
print("\nColumns:")
for column in parquet_columns:
    print(f"- {column}")

In [ ]:
NMA_COLUMNS = [
    "stimulus_presentations_id",
    "cell_specimen_id",
    "trace",
    "trace_timestamps",
    "mean_response",
    "baseline_response",
    "image_name",
    "image_index",
    "is_change",
    "omitted",
    "mean_running_speed",
    "mean_pupil_area",
    "response_latency",
    "rewarded",
    "ophys_experiment_id",
    "imaging_depth",
    "targeted_structure",
    "cre_line",
    "session_type",
    "session_number",
    "mouse_id",
    "ophys_session_id",
    "ophys_container_id",
    "behavior_session_id",
    "full_genotype",
    "reporter_line",
    "driver_line",
    "indicator",
    "sex",
    "age_in_days",
    "exposure_level",
]

missing_columns = sorted(set(NMA_COLUMNS) - set(parquet_columns))
extra_columns = sorted(set(parquet_columns) - set(NMA_COLUMNS))

print("Missing Neuromatch-compatible columns:", missing_columns)
print("Extra Allen-derived columns:", extra_columns)

if missing_columns:
    raise ValueError(f"Parquet is missing expected column(s): {missing_columns}")

In [ ]:
data = pd.read_parquet(PARQUET_PATH)

summary = pd.Series(
    {
        "rows": len(data),
        "columns": len(data.columns),
        "cells": data["cell_specimen_id"].nunique(dropna=True),
        "stimuli": data["stimulus_presentations_id"].nunique(dropna=True),
        "experiments": data["ophys_experiment_id"].nunique(dropna=True),
        "sessions": data["ophys_session_id"].nunique(dropna=True),
        "mice": data["mouse_id"].nunique(dropna=True),
        "cre_lines": ", ".join(sorted(map(str, data["cre_line"].dropna().unique()))),
    }
)

display(summary.to_frame("value"))
display(data.head())

In [ ]:
count_tables = []
for column in ["cre_line", "targeted_structure", "session_type", "exposure_level", "is_change", "omitted", "rewarded"]:
    counts = data[column].value_counts(dropna=False).rename("count").reset_index()
    counts.columns = [column, "count"]
    count_tables.append((column, counts))

for column, counts in count_tables:
    print(f"\n{column}")
    display(counts)

In [ ]:
def to_float_array(value):
    return np.asarray(value, dtype=float)


plot_data = data.dropna(subset=["trace", "trace_timestamps"]).copy()
preferred = plot_data[plot_data["is_change"].fillna(False).astype(bool)]
if preferred.empty:
    preferred = plot_data

cell_ids = preferred["cell_specimen_id"].dropna().drop_duplicates().head(4).tolist()
fig, axes = plt.subplots(len(cell_ids), 1, figsize=(9, 2.4 * max(len(cell_ids), 1)), sharex=True)
if len(cell_ids) == 1:
    axes = [axes]

for ax, cell_id in zip(axes, cell_ids):
    row = preferred[preferred["cell_specimen_id"] == cell_id].iloc[0]
    trace = to_float_array(row["trace"])
    timestamps = to_float_array(row["trace_timestamps"])
    ax.plot(timestamps, trace, lw=1.5)
    ax.axvline(0, color="black", ls="--", lw=1)
    ax.set_ylabel("dF/F")
    ax.set_title(
        f"cell {cell_id} | image={row.get('image_name')} | "
        f"change={row.get('is_change')} | omitted={row.get('omitted')}"
    )

axes[-1].set_xlabel("time from stimulus onset (s)")
fig.tight_layout()
plt.show()

In [ ]:
response_summary = (
    data.groupby(["cre_line", "is_change", "omitted"], dropna=False)["mean_response"]
    .agg(["count", "mean", "sem"])
    .reset_index()
)
display(response_summary)

labels = response_summary.apply(
    lambda row: f"{row['cre_line']}\nchange={row['is_change']}\nomitted={row['omitted']}",
    axis=1,
)

fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(response_summary)), 4))
ax.bar(labels, response_summary["mean"], yerr=response_summary["sem"], capsize=3)
ax.set_ylabel("mean response")
ax.set_title("Mean response by stimulus condition")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()